In [1]:
%pip install crewai langchain langchain-openai langchain-community langchain-tavily tavily-python pydantic 
%pip install litellm

INFO: pip is looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ------------------------------- -------- 0.8/1.0 MB 4.2 MB/s eta 0:00:01
   ---------------------------------------- 1.0/1.0 MB 3.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ------------- -------------------------- 0.8/2.4 MB 3.7 MB/s eta 0:00:01
   -------------------------- ------------- 1.6/2.4 MB 4.0 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 4.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   --------------- ------------------------ 0.8/2.0 MB 3.

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

   ---------------------------------------- 0.0/15.5 MB ? eta -:--:--
   -- ------------------------------------- 0.8/15.5 MB 4.2 MB/s eta 0:00:04
   ---- ----------------------------------- 1.6/15.5 MB 4.2 MB/s eta 0:00:04
   ------ --------------------------------- 2.4/15.5 MB 4.1 MB/s eta 0:00:04
   -------- ------------------------------- 3.1/15.5 MB 4.0 MB/s eta 0:00:04
   ---------- ----------------------------- 4.2/15.5 MB 4.0 MB/s eta 0:00:03
   ------------ --------------------------- 5.0/15.5 MB 4.0 MB/s eta 0:00:03
   -------------- ------------------------- 5.8/15.5 MB 4.1 MB/s eta 0:00:03
   ---------------- ----------------------- 6.6/15.5 MB 4.0 MB/s eta 0:00:03
   ------------------- -------------------- 7.6/15.5 MB 4.0 MB/s eta 0:00:02
   --------------------- ------------------ 8.4/15.5 MB 4.0 MB/s eta 0:00:02
   ----------------------- ---------------- 9.2/15.5 MB 4.0 MB/s eta 0:00:02
   ------------------------- -------------- 10.0/15.5 MB 4.1 MB/s eta 0:00:02
   --

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import requests
from crewai.llm import LLM
from crewai import Agent, Task, Crew, Process
from crewai.tools import BaseTool
from dotenv import load_dotenv
from langchain_core.tools import tool

load_dotenv()


ModuleNotFoundError: No module named 'pywintypes'

In [ ]:
router_agent = Agent(
    role="Request Router",
    goal="""
    Determine whether the user request is related to:
    - Product information
    - Company information

    Return ONLY:
    PRODUCT
    or
    COMPANY
    """,
    backstory="Expert request classifier.",
    verbose=True,
)

product_retriever = Agent(
    role="Product Retriever",
    goal="Retrieve detailed information about products.",
    backstory="Specialized in product search and retrieval.",
    verbose=True,
)

company_retriever = Agent(
    role="Company Retriever",
    goal="Retrieve detailed information about companies.",
    backstory="Specialized in company research.",
    verbose=True,
)

# -------------------------
# Router Task
# -------------------------

user_request = "Tell me about the latest iPhone"

routing_task = Task(
    description=f"""
    Classify the following request:

    {user_request}

    Return only PRODUCT or COMPANY.
    """,
    expected_output="PRODUCT or COMPANY",
    agent=router_agent,
)

# Execute router first
router_crew = Crew(
    agents=[router_agent],
    tasks=[routing_task],
    process=Process.sequential,
)

route = router_crew.kickoff().raw.strip().upper()

print(f"Route selected: {route}")

# -------------------------
# Conditional Routing
# -------------------------

if route == "PRODUCT":

    retrieval_task = Task(
        description=f"""
        Retrieve detailed product information for:

        {user_request}
        """,
        expected_output="Detailed product information",
        agent=product_retriever,
    )

    crew = Crew(
        agents=[product_retriever],
        tasks=[retrieval_task],
        process=Process.sequential,
    )

elif route == "COMPANY":

    retrieval_task = Task(
        description=f"""
        Retrieve detailed company information for:

        {user_request}
        """,
        expected_output="Detailed company information",
        agent=company_retriever,
    )

    crew = Crew(
        agents=[company_retriever],
        tasks=[retrieval_task],
        process=Process.sequential,
    )

else:
    raise ValueError(f"Unknown route: {route}")

result = crew.kickoff()

print(result)